In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "01B_GSE135779_CHILD_CELLTYPE_LABELLING")


In [ ]:
#LOAD PREVIOUS STAGE OUTPUT

import scanpy as sc

adata = sc.read_h5ad(f"{BASE_DIR}/child_individual_h5ad/adata_child_processed_leiden08.h5ad")

## Cell-type annotation

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
marker_table = sc.get.rank_genes_groups_df(adata, group=None)
marker_table.to_csv(FIGURE_DIR / "marker_genes_all_clusters.csv", index=False)
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, save="_figure_01.png")

**Reviewed cluster-to-cell-type map.** The final labels use the independently reviewed map supplied for the current Leiden solution and are supported by the complete marker table saved as `marker_genes_all_clusters.csv`. Clusters 0 and 1 are ribosomal/low-quality; cluster 2 is classical monocytes; cluster 3 is B cells; cluster 4 is CD4 T cells; cluster 5 is CD8 T cells; cluster 6 is NK cells; cluster 7 is non-classical monocytes; cluster 8 is cDCs; cluster 9 is RBCs; cluster 10 is platelets; cluster 11 is plasma cells; cluster 12 is IFN-stimulated T cells; and cluster 13 is pDCs. Ribosomal/low-quality, erythroid, and platelet clusters are excluded from primary communication inference. CellTypist remains a supporting cross-check rather than proof of a label.

In [ ]:
# CELL TYPE MAP

cell_type_map = {
    "0": "Ribosomal/Low-quality",
    "1": "Ribosomal/Low-quality",
    "2": "Classical Monocytes",
    "3": "B Cells",
    "4": "CD4 T Cells",
    "5": "CD8 T Cells",
    "6": "NK Cells",
    "7": "Non-classical Monocytes",
    "8": "cDCs",
    "9": "RBCs",
    "10": "Platelets",
    "11": "Plasma Cells",
    "12": "IFN-stimulated T Cells",
    "13": "pDCs"
}

**Symmetric primary-analysis rule.** Child and adult cohorts use the same cluster-level annotation procedure. Clusters dominated by technical ribosomal or mitochondrial signatures without a clear lineage identity are excluded in both cohorts. No cohort-specific post hoc subclustering or manual rescue is used in the primary cell-cell communication analysis. This conservative rule prevents differential annotation effort from creating an apparent age-group difference.

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden"].map(cell_type_map)
print(adata.obs["cell_type"].value_counts())

In [ ]:
sc.pl.umap(adata, color="cell_type", legend_loc="on data", save="_figure_02.png")

In [ ]:
bad_types = [
    "Ribosomal/Low-quality",
    "Mitochondrial-high/Low-quality",
    "Mixed platelet-monocyte/Low-quality",
    "RBCs",
    "Platelets",
]

adata_clean = adata[~adata.obs["cell_type"].isin(bad_types)].copy()

print("Before:", adata.shape)
print("After:", adata_clean.shape)
print(adata_clean.obs["cell_type"].value_counts())
from qc_utils import export_annotation_evidence
annotation_audit = export_annotation_evidence(
    adata, cell_type_map, bad_types, "child", BASE_DIR, marker_table=marker_table
)


In [ ]:
sc.pl.umap(adata_clean, color="cell_type", legend_loc="on data", save="_figure_03.png")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

adata.obs["leiden_res_0.8"] = adata.obs["leiden_res_0.8"].astype(str)

# UMAP with cluster numbers on the plot
sc.pl.umap(
    adata,
    color="leiden_res_0.8",
    legend_loc="on data",
    legend_fontsize=12,
    legend_fontweight="bold",
    size=3,
    frameon=True,
    title="Leiden clusters"
, save="_figure_04.png")

# print number legend as a clean table
legend_df = pd.DataFrame({
    "cluster": list(cell_type_map.keys()),
    "cell_type": list(cell_type_map.values())
}).sort_values("cluster", key=lambda x: x.astype(int))

legend_df

In [ ]:
adata_clean.write_h5ad(
    f"{BASE_DIR}/child_individual_h5ad/adata_child_final_liana.h5ad"
)

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np

adata = sc.read_h5ad(f"{BASE_DIR}/child_individual_h5ad/adata_child_processed_leiden08.h5ad")

# subsample ~50k cells (seeded for reproducibility)
from analysis_config import CELLTYPIST_MAX_CELLS
n_sub = min(CELLTYPIST_MAX_CELLS, adata.n_obs)
rng = np.random.default_rng(0)
idx = np.sort(rng.choice(adata.n_obs, n_sub, replace=False))
import gc
gc.collect()
# Build a lightweight object without duplicating .raw and the counts layer.
# CellTypist needs the log-normalized expression matrix and observation labels only.
adata_sub = ad.AnnData(
    X=adata.X[idx].copy(),
    obs=adata.obs.iloc[idx].copy(),
    var=adata.var.copy(),
)

print("Subsample:", adata_sub)

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, BASE_DIR + '/Notebooks')
from sample_lists import CHILD_HEALTHY_SAMPLES, CHILD_SLE_SAMPLES, ADULT_HEALTHY_SAMPLES, ADULT_SLE_SAMPLES

# Load AnnData
adata = sc.read_h5ad(
    f"{BASE_DIR}/child_individual_h5ad/adata_child_final_liana.h5ad"
)

# Lists
healthy_samples = CHILD_HEALTHY_SAMPLES

sle_samples = CHILD_SLE_SAMPLES

# ----- CHANGE "sample" if needed -----
sample_col = "sample"

# Create condition labels
adata.obs["condition"] = "Unknown"

adata.obs.loc[
    adata.obs[sample_col].isin(healthy_samples),
    "condition"
] = "Healthy"

adata.obs.loc[
    adata.obs[sample_col].isin(sle_samples),
    "condition"
] = "SLE"

# Remove unknown cells if any
adata = adata[adata.obs["condition"] != "Unknown"].copy()

# UMAP is already present in adata_child_final_liana.h5ad (computed on the
# Harmony batch-corrected embedding in 1A). Do not recompute it here -- a naive
# re-normalize + non-batch-corrected PCA/UMAP would both double-transform
# already-log-normalized data and silently discard the Harmony correction.
assert "X_umap" in adata.obsm, "Expected UMAP to already be present from 1A/upstream processing."

# Plot
sc.pl.umap(
    adata,
    color="condition",
    title="Healthy vs SLE UMAP",
    frameon=False,
    size=20
, save="_figure_05.png")

In [ ]:
# CELL TYPIST

import celltypist

predictions = celltypist.annotate(
    adata_sub,
    model="Immune_All_Low.pkl",
    majority_voting=True
)

adata_sub.obs = adata_sub.obs.join(predictions.predicted_labels)

In [ ]:
import pandas as pd

celltypist_cluster_agreement = pd.crosstab(
    adata_sub.obs["leiden_res_0.8"],
    adata_sub.obs["majority_voting"],
    normalize="index"
).round(2)
celltypist_cluster_agreement.to_csv(
    f"{BASE_DIR}/Results/qc/child_celltypist_cluster_agreement.csv"
)
display(celltypist_cluster_agreement)


**Interpreting the CellTypist cross-check.** The crosstab reports automated labels by Leiden cluster and is evaluated together with the exported marker table. Agreement supports—but does not prove—the manual broad-lineage labels. Disagreement, mixed predictions, or weak canonical-marker expression is retained as an annotation limitation and must be reviewed before manuscript submission.

In [ ]:
# Donor-by-cell-type abundance table for annotation and composition QC.
from qc_utils import (export_celltype_counts, export_marker_validation, export_integration_diagnostics)

celltype_counts = export_celltype_counts(adata_clean, f"{BASE_DIR}/Results/qc/child_sample_celltype_counts.csv")
display(celltype_counts.head())

marker_validation = export_marker_validation(adata_clean, f"{BASE_DIR}/Results/qc/child_canonical_marker_validation.csv")
integration_qc = export_integration_diagnostics(adata_clean, f"{BASE_DIR}/Results/qc/child_harmony_diagnostics.csv")
display(marker_validation)
display(integration_qc)
